# Deep Research — Reference Guide

A multi-agent pipeline that plans web searches, runs them in parallel, writes a long-form report, and emails the result.

**Architecture pattern:** orchestrate with *your* Python code (not an LLM manager). Call `Runner.run()` once per step, use structured outputs between steps, and keep each agent narrowly scoped.

| Agent | Role | Key SDK feature |
|-------|------|-----------------|
| Planner | Turns a query into N search terms | `output_type=` (structured output) |
| Search | Summarizes one web search | Hosted `WebSearchTool` + `tool_choice="required"` |
| Writer | Turns search summaries into a report | `output_type=ReportData` |
| Email | Formats HTML and sends | `@function_tool` custom tool |

**Pipeline:** `query → plan searches → parallel search → write report → email`

In [1]:
# --- Core OpenAI Agents SDK ---
# Agent:       defines an LLM worker (instructions + optional tools / output schema)
# Runner:      executes an Agent for a given input; await Runner.run(agent, input)
# WebSearchTool: OpenAI-hosted tool — the model can call it to search the web
# function_tool: decorator that turns a Python function into a tool the agent can call
# trace:       context manager that groups a multi-step run in the OpenAI Traces UI
from agents import Agent, WebSearchTool, trace, Runner, function_tool

# ModelSettings: per-agent knobs (e.g. force tool use with tool_choice="required")
from agents.model_settings import ModelSettings

# Pydantic BaseModel + Field: define structured output schemas the LLM must return
from pydantic import BaseModel, Field

# load_dotenv: pull OPENAI_API_KEY (and email creds) from a local .env file
from dotenv import load_dotenv
import asyncio  # used later to run multiple search agents in parallel (asyncio.gather)
from IPython.display import display, Markdown  # render agent markdown output in the notebook

# Notebook is in my-work/; course messenger.py lives one level up in 2_openai/.
# Without this, Python may import an unrelated PyPI/site-packages "messenger" package.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))
from messenger import send_email, push  # send_email → real email; push → Pushover fallback

In [2]:
# override=True: values in .env win over any already-set environment variables.
# Returns True if a .env file was found and loaded.
load_dotenv(override=True)

True

In [3]:
# --- Tunables for this lab ---
MODEL_NAME = "gpt-5.4-mini"  # model used by every agent below

# True  → Email Agent calls send_email() (needs email credentials in .env)
# False → falls back to push() / Pushover so you can still test the pipeline
USE_EMAIL = True

# How many distinct web searches the Planner Agent should propose.
# Each search ≈ one WebSearchTool call (~$0.01+), so this drives cost.
HOW_MANY_SEARCHES = 5

## Strategy: orchestrate with code (the "bulletproof" way)

Two common multi-agent patterns:

1. **LLM-as-manager** — one agent decides which specialist to call next.
2. **Code-as-orchestrator** — *your* Python decides the order; each agent does one job.

This lab uses **#2**. Benefits:

- Deterministic pipeline (plan → search → write → email)
- Easy to debug: each `Runner.run()` is a clear boundary
- Structured outputs between steps (typed Python objects, not free-form text)
- Parallelism where it helps (`asyncio.gather` for searches)

### The 4 agents

1. **Search Agent** — given one search term, uses `WebSearchTool` and returns a short summary
2. **Planner Agent** — given a research question, returns a `WebSearchPlan` (list of queries + reasons)
3. **Writer Agent** — given the query + search summaries, returns a `ReportData` object
4. **Email Agent** — given the markdown report, calls `send_email_tool` with HTML

Then four thin async Python functions wrap `Runner.run()` for each agent so the final script reads like a workflow.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

Docs: https://openai.github.io/openai-agents-python/tools/#hosted-tools

Hosted tools run on OpenAI's infrastructure. Convenient, but they cost money and lock you into OpenAI.

| Tool | Purpose |
|------|---------|
| `WebSearchTool` | Search the live web |
| `FileSearchTool` | Retrieve from OpenAI Vector Stores |
| `CodeInterpreterTool` | Run code in a sandbox |
| `HostedMCPTool` | Expose a remote MCP server's tools |
| `ImageGenerationTool` | Generate images from a prompt |
| `ToolSearchTool` | Lazily load deferred tools / MCP servers |

### Cost note — `WebSearchTool`

Roughly **~$0.01 per call** (sometimes more if OpenAI bills multiple searches inside one call). With `HOW_MANY_SEARCHES = 5`, a full pipeline run can be ~$0.05+ just for search, and the next labs stack more of this. Skip the live runs if cost is a concern; later labs cover cheaper search options.

Pricing: https://developers.openai.com/api/docs/pricing

In [4]:
# System prompt for the Search Agent: one job — search + short summary, nothing else.
# "Reply only with the summary" keeps final_output clean for later concatenation.
INSTRUCTIONS = """ 
You are a research assistant. Given a search term, you search
the web for that term and produce a concise summary of the 
results. The summary must be 2-3 paragraphs and less than 300
words. Capture the main points and be succinct. Reply only with
the summary. 
"""

# Sample research question used when testing agents individually,
# and again as the end-to-end query in the orchestration section.
task = "Most popular AI Agent frameworks in 2026"

# tool_choice="required" → the model MUST call a tool (WebSearchTool) before answering.
# Without this, it might skip search and invent an answer from training data.
settings = ModelSettings(tool_choice="required")

# Hosted web search — OpenAI runs the search; the agent receives results as tool output.
tools = [WebSearchTool()]

In [5]:
# Build the Search Agent.
# - name:           label in traces / logs
# - instructions:   system prompt (role + output constraints)
# - tools:          what the agent is allowed to call (WebSearchTool)
# - model:          which LLM backs this agent
# - model_settings: force tool use so it cannot skip searching
search_agent = Agent(
    name="Search Agent",
    instructions=INSTRUCTIONS,
    tools=tools,
    model=MODEL_NAME,
    model_settings=settings,
)

In [6]:
# Smoke-test the Search Agent alone (before wiring it into the full pipeline).
# Runner.run is async → must await in a notebook / async context.
# result.final_output is a string here (no output_type), so we render it as Markdown.
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

# Tip: inspect the full tool-call chain at https://platform.openai.com/traces

In 2026, the most commonly cited AI agent frameworks are **LangGraph, CrewAI, AutoGen, and OpenAI’s Agents SDK**. Recent comparison pieces and ecosystem analyses consistently place these at the center of the landscape, with **LangGraph** often described as the strongest choice for complex, stateful, production workflows; **CrewAI** as popular for quick multi-agent prototypes; **AutoGen** for conversational or research-style agents; and **OpenAI Agents SDK** for OpenAI-native builds. ([the-agent-report.com](https://the-agent-report.com/2026/07/ai-agent-frameworks-comparison-2026-langgraph-crewai-autogen/?utm_source=openai))

Popularity signals are imperfect, but GitHub activity and ecosystem studies suggest **LangGraph has especially strong momentum**, while **CrewAI and AutoGen remain widely used** and **OpenAI Agents SDK has grown quickly since its 2025 release**. Broader studies also caution that GitHub stars alone are not a reliable quality measure, so the “most popular” framework depends on whether you mean adoption, community buzz, or production usage. ([github.com](https://github.com/langchain-ai/langgraph?utm_source=openai))

If you want, I can also give you a **ranked top 10** or a **“best framework by use case”** list for 2026.

## Agent 2: The Planner Agent

The Planner does **not** search the web. It only decides *what* to search.

**Structured Outputs** (`output_type=SomePydanticModel`) force the LLM to return JSON that matches your schema. The SDK parses it into a real Python object (`result.final_output` is a `WebSearchPlan`, not a string).

Pattern used here:

- `WebSearchItem` — one planned search (`reason` + `query`)
- `WebSearchPlan` — container with `searches: list[WebSearchItem]`

`Field(description=...)` text is sent to the model as guidance for what each field means — treat descriptions as part of the prompt.

In [7]:
# One planned search: why we need it + the exact query string to run.
class WebSearchItem(BaseModel):
    reason: str = Field(
        description="Your reasoning for why this search is important to the query."
    )
    query: str = Field(
        description="The search term to use for the web search."
    )


# Top-level structured output for the Planner Agent.
# Use list[WebSearchItem] (brackets), NOT list(WebSearchItem) (parentheses).
# Parentheses call list() and try to iterate the class → TypeError.
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="A list of web searches to perform to best answer the query"
    )

In [8]:
# Inspect the JSON Schema the SDK derives from your Pydantic models.
# This (plus Field descriptions) is what the model is constrained to produce.
# Useful for debugging: if a field is wrong here, the agent output will be wrong too.
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [9]:
# f-string injects HOW_MANY_SEARCHES so the prompt and your constant stay in sync.
# Note: Planner has NO tools — it only reasons and returns structured JSON.
INSTRUCTIONS = f""" 
You are a research assistant. Given a user query, come up with
a set of web searches to perform to best answer the query. Output
{HOW_MANY_SEARCHES} terms to query for.
"""

# output_type=WebSearchPlan → result.final_output will be a WebSearchPlan instance
# (access .searches, .searches[0].query, etc.) instead of a raw string.
planner_agent = Agent(
    name="Planner Agent",
    instructions=INSTRUCTIONS,
    model=MODEL_NAME,
    output_type=WebSearchPlan,
)

In [10]:
# Smoke-test the Planner. With output_type set, final_output is already parsed.
# model_dump_json() → pretty-print the structured plan as JSON for inspection.
result = await Runner.run(planner_agent, task)
print(result.final_output.model_dump_json(indent=2))

# In the full pipeline you would iterate:
#   for item in result.final_output.searches:
#       ... use item.query and item.reason ...

{
  "searches": [
    {
      "reason": "Find current rankings, surveys, and ecosystem analyses on AI agent frameworks in 2026.",
      "query": "most popular AI agent frameworks 2026 ranking"
    },
    {
      "reason": "Capture market adoption and community interest signals for agent frameworks.",
      "query": "AI agent frameworks popularity survey 2026"
    },
    {
      "reason": "Identify widely used open-source and enterprise agent frameworks with recent traction.",
      "query": "best AI agent frameworks 2026 open source enterprise"
    },
    {
      "reason": "Check developer discussion and GitHub activity for leading frameworks.",
      "query": "AI agent framework GitHub stars 2026"
    },
    {
      "reason": "Compare major frameworks mentioned in recent benchmarks, articles, and vendor docs.",
      "query": "top AI agent frameworks comparison 2026"
    }
  ]
}


## Agent 3: The Writer Agent

Takes the original query **plus** the search summaries and produces a long-form report.

Again uses structured output (`ReportData`) so the pipeline can reliably grab:

- `short_summary` — brief executive blurb
- `markdown_report` — the full document (passed to the Email Agent)
- `follow_up_questions` — suggested next research topics

In the smoke-test below we only pass `task` (no search results). In the real pipeline, `write_report()` injects both query and search summaries into the input message.

In [11]:
# Writer prompt: emphasize length/detail so the report is useful, not a brief.
INSTRUCTIONS = """ 
You are a senior researcher tasked with writing a cohesive 
report for a research query. You will be provided with the
original query, and some research. Generate a comprehensive
report based on the research and the query. The final output
should be in markdown format, and it should be lengthy and
detailed. Aim for 5-10 pages of content, at least 1000 words.
"""


# Structured report: three fields the orchestrator (or UI) can use independently.
class ReportData(BaseModel):
    short_summary: str = Field(
        description="A short 2-3 sentence summary of the findings."
    )
    markdown_report: str = Field(
        description="The final report"
    )
    follow_up_questions: list[str] = Field(
        description="Suggested topics to research further"
    )


# No tools — pure generation into a typed schema.
writer_agent = Agent(
    name="Writer Agent",
    instructions=INSTRUCTIONS,
    model=MODEL_NAME,
    output_type=ReportData,
)

In [12]:
# Smoke-test Writer alone (no real search context here — just the query).
# In production, prefer write_report(query, search_results) so the model
# grounds the report in the Search Agent summaries.
result = await Runner.run(writer_agent, task)
print(result.final_output.model_dump_json(indent=2))
# Useful fields: result.final_output.short_summary
#                result.final_output.markdown_report
#                result.final_output.follow_up_questions

{
  "short_summary": "As of 2026, the AI agent framework landscape is best understood as a mix of general-purpose orchestration tools, enterprise workflow platforms, and code-first agent SDKs. The most popular options are driven less by hype alone and more by developer adoption, ecosystem maturity, and ease of integration with LLMs, tools, and memory systems.",
  "markdown_report": "# Most Popular AI Agent Frameworks in 2026\n\nAI agent frameworks in 2026 sit at the intersection of orchestration, tool use, memory, planning, evaluation, and deployment. The term “agent framework” now covers a broad range of products and libraries: some help developers build autonomous or semi-autonomous LLM-powered agents from scratch, while others provide workflow graphs, enterprise guardrails, or low-code interfaces for business automation. Popularity in this space is shaped by a few consistent factors: model compatibility, reliability, observability, production readiness, community adoption, and how q

## Agent 4: The Email Agent

Converts the markdown report into a polished HTML email and sends it via a **custom function tool**.

### `@function_tool` vs hosted tools

| | Hosted (`WebSearchTool`) | Custom (`@function_tool`) |
|--|--------------------------|---------------------------|
| Runs where? | OpenAI cloud | Your Python process |
| Schema from? | Built-in | Function signature + docstring |
| Use for | Search, code exec, etc. | App logic: email, DB, APIs |

The decorator inspects the function signature and docstring to build a JSON schema the model fills in when it calls the tool. The agent decides *when* to call it; your code decides *what* happens (email vs Pushover).

In [13]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all
    sales prospects.

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    # Docstring + type hints above become the tool schema the LLM sees.
    # Return value is fed back to the model as the tool result.
    if USE_EMAIL:
        send_email(subject, text_body, html_body)  # real email via messenger.py
    else:
        push(f"Subject: {subject}\n\n{text_body}")  # cheap/safe fallback for testing
    return "Email sent successfully"

In [14]:
# Schema the model sees for this tool — derived from the function signature.
# Compare to WebSearchPlan.model_json_schema(): same idea (JSON Schema),
# but for a *tool call* rather than the agent's final output.
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [15]:
# Email Agent: given a report, must use send_email_tool to deliver HTML email.
# No output_type — we care that the tool ran successfully, not about a typed final answer.
INSTRUCTIONS = """ 
You are provided with a detailed report. Use your tool to send
an email, converting the report into a clean, well presented
HTML email with an appropriate subject line.
"""

email_agent = Agent(
    name="Email Agent",
    instructions=INSTRUCTIONS,
    tools=[send_email_tool],
    model=MODEL_NAME,
)

## Orchestrate by code

Agents are defined above; now Python owns the workflow.

```
query
  → Planner Agent  → WebSearchPlan
  → N × Search Agent (in parallel via asyncio.gather)
  → Writer Agent   → ReportData
  → Email Agent    → send_email_tool(...)
```

Each helper below is a thin wrapper around `Runner.run()` so the final cell reads like a script, not nested agent logic.

In [16]:
async def run_searches(query: str):
    """Plan searches with the Planner, then run all Search Agents in parallel."""
    print("Planning searches...")
    # Planner returns WebSearchPlan; .searches is list[WebSearchItem]
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")

    # Create one coroutine per search item (does not run them yet).
    tasks = [search(item) for item in searches]

    # Run all searches concurrently — major speedup vs sequential awaits.
    # Cost still scales with HOW_MANY_SEARCHES (each call uses WebSearchTool).
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results  # list[str] — one summary per search


async def search(item: WebSearchItem):
    """Run the Search Agent for a single planned query."""
    # Pass both query and reason so the summarizer knows what angle to emphasize.
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output  # plain string summary (no output_type on Search Agent)

In [17]:
async def write_report(query: str, search_results: list[str]):
    """Feed the original query + all search summaries into the Writer Agent."""
    print("Thinking about report...")
    # Bundle everything into one user message; Writer has no tools — it just synthesizes.
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output  # ReportData (short_summary, markdown_report, follow_up_questions)


async def send_report_email(report: ReportData):
    """Hand the markdown report to the Email Agent, which must call send_email_tool."""
    print("Writing email...")
    # Only the markdown body is needed; the agent invents subject + HTML layout.
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

In [18]:
# --- End-to-end Deep Research run ---
query = "Most popular AI Agent frameworks in 2026"

# trace(...) groups every nested Runner.run under one trace name in the OpenAI UI.
# Open https://platform.openai.com/traces after this cell to inspect the full graph:
# Planner → 5 Searches → Writer → Email (with tool call).
with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)       # plan + parallel search
    report = await write_report(query, search_results)  # ReportData
    await send_report_email(report)                  # side effect: email / push
    print("Hooray!!!")

# Optional: peek at pieces of the report in-notebook
# display(Markdown(report.short_summary))
# display(Markdown(report.markdown_report))
# print(report.follow_up_questions)

Starting research...
Planning searches...
Will perform 5 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!!!


In [19]:
print(f"Report: {report.markdown_report}")

Report: # Most Popular AI Agent Frameworks in 2026

## Executive Summary

The AI agent framework landscape in 2026 is crowded, fast-moving, and still in the process of consolidating. Unlike earlier software categories where one framework quickly became the default, AI agent frameworks have split into several subcategories based on how teams build agents: stateful graph orchestration, role-based multi-agent coordination, SDK-native provider stacks, and enterprise-centric tooling. As a result, the answer to “what are the most popular frameworks?” is not a single winner, but a short list of frameworks that consistently appear across comparisons, community discussions, enterprise evaluations, and vendor-aligned recommendations.

Across the research, the same names surface repeatedly: **LangGraph/LangChain**, **CrewAI**, **AutoGen/AG2**, **Semantic Kernel**, **LlamaIndex**, and the **OpenAI Agents SDK**. Additional frameworks and platforms with meaningful momentum include **Google ADK**, **

## Quick reference card

### SDK building blocks used here

| Concept | How it appears in this notebook |
|---------|----------------------------------|
| `Agent(...)` | One specialist: instructions + model + optional tools / `output_type` |
| `Runner.run(agent, input)` | Execute one agent turn; always `await` |
| `output_type=Model` | Structured final answer → Pydantic instance on `result.final_output` |
| `tools=[...]` | Hosted (`WebSearchTool`) or custom (`@function_tool`) |
| `ModelSettings(tool_choice="required")` | Force the model to call a tool before answering |
| `trace("name")` | Bundle multi-step runs for the Traces dashboard |
| `asyncio.gather` | Parallelize independent agent calls (the searches) |

### Design takeaways

1. **Narrow agents** — each has one job and a clear I/O contract.
2. **Orchestrate in code** — Python owns sequencing; LLMs own reasoning inside each step.
3. **Structure the handoffs** — Pydantic schemas between Planner → Search → Writer.
4. **Parallelize independent work** — searches fan out; write/email stay sequential.
5. **Watch cost** — `HOW_MANY_SEARCHES` × WebSearchTool price dominates spend.

### Inspect after a run

- Traces: https://platform.openai.com/traces
- Email / Pushover depending on `USE_EMAIL`